In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from netCDF4 import Dataset
import os, fnmatch
import datetime
from mpl_toolkits.basemap import Basemap
import pickle

In [ ]:
def find_files(directory, pattern, maxdepth=None):
    flist = []
    for root, dirs, files in os.walk(directory):
        for basename in files:
            if fnmatch.fnmatch(basename, pattern):
                filename = os.path.join(root, basename)
                filename = filename.replace('\\\\', os.sep)
                if maxdepth is None:
                    flist.append(filename)
                else:
                    if filename.count(os.sep)-directory.count(os.sep) <= maxdepth:
                        flist.append(filename)
    return flist

In [ ]:
def select_data(longitude, latitude, sss, borders):
    n, s, w, e, = borders
    mask = (latitude >= s) & (latitude <= n) & (longitude >= w) & (longitude <= e)
    
    rows_with_data = np.any(mask, axis=1)
    cols_with_data = np.any(mask, axis=0)

    row_idx = np.where(rows_with_data)[0]
    col_idx = np.where(cols_with_data)[0]

    lon = longitude[row_idx.min():row_idx.max() + 1,
                    col_idx.min():col_idx.max() + 1]
    
    lat = latitude[row_idx.min():row_idx.max() + 1,
               col_idx.min():col_idx.max() + 1]
    
    sss_ = sss[row_idx.min():row_idx.max() + 1,
            col_idx.min():col_idx.max() + 1]
    
    return lon, lat, sss_

In [ ]:
def make_data(f, borders):
    n, s, w, e, = borders
    data = Dataset(f, 'r')
    
    latitude = np.asarray(data['lat'])
    longitude = np.asarray(data['lon'])
    sss = np.asarray(data['sss'])

    data.close()
    
    lon, lat, sss_ = select_data(longitude=longitude, latitude=latitude, sss=sss, borders=borders)
    sss_ = np.where((lon < w) | (lon > e) | (lat > n) | (lat < s), np.nan, sss_)

    return lon, lat, sss_

In [ ]:
files = find_files('/mnt/hippocamp/asavin/data/SSS_ESACCI_grid/SSS_ESACCI_grid_data', '*.nc')
files.sort()
files

In [ ]:
files[-1]

In [ ]:
len(files)

In [ ]:
array = []

for file in files:
    _, _, sss = make_data(file, borders=[80,70,55,105])
    array.append(sss)

In [ ]:
len(array)

In [ ]:
array[0].shape

In [ ]:
array_upd = np.stack(array, axis=0)
array_upd.shape

In [ ]:
np.nanmin(array_upd), np.nanmax(array_upd)

In [ ]:
plt.hist(array_upd.ravel(), bins=100);

In [ ]:
plt.hist(((array_upd - np.nanmean(array_upd)) / np.nanstd(array_upd)).ravel(), bins=100);

In [ ]:
inv_array = np.where(array_upd != 0, 1 / array_upd, np.nan)

In [ ]:
plt.hist(inv_array.ravel(), bins=5000);

In [ ]:
plt.hist(((inv_array - np.nanmean(inv_array)) / np.nanstd(inv_array)).ravel(), bins=500);

In [ ]:
array_log = np.log(inv_array)

In [ ]:
plt.hist(array_log.ravel(), bins=100);

In [ ]:
np.nanmin(array_log)

In [ ]:
array_log_pos = array_log + 4

In [ ]:
plt.hist(array_log_pos.ravel(), bins=100);

In [ ]:
np.nanmedian(array_log_pos)

In [ ]:
array_log2 = np.arcsinh(array_log/0.6)

In [ ]:
plt.hist(array_log2.ravel(), bins=500);

In [ ]:
plt.hist(((array_log2 - np.nanmean(array_log2)) / np.nanstd(array_log2)).ravel(), bins=5000);

In [ ]:
np.nanmean(array_log2), np.nanstd(array_log2)

In [ ]:
np.nanmax(array_log2)